In [1]:
from tkinter import *
from tkinter import messagebox
from random import sample

class mineSweeper(Frame):
    def __init__(self, rows, cols):
        Frame.__init__(self)
        self.rows = rows
        self.cols = cols
        self.mineFields = []
        self.mines = []
        self.count = 0
        # initialize the mineFields 2D list (row x col) with each cell a list
        # containing a button and a number indicating the number of mines a round it.
        self.retryBtn = Button(self, text="Retry", command=self.restart)
        self.retryBtn.grid(row=self.rows, column=0, columnspan=self.cols//2, sticky="ew")

        self.quitBtn = Button(self, text="Quit", command=self.winfo_toplevel().destroy)
        self.quitBtn.grid(row=self.rows,column=self.cols//2,columnspan=self.cols-self.cols//2,sticky="ew")
        
        self.neighbours =  [
                [-1,-1],[-1,0],[-1,1],
                [0,-1],        [0,1],
                [1,-1],[1,0],[1,1]
            ]
        for r in range(rows):
            temp = []
            for c in range(cols):
                b = Button(self, height=1, width=2)
                if (r+c)%2 == 0:
                    # print(cols%2,r)
                    b.config(bg = "DarkOliveGreen2") #1,3,5 (even rows)
                else:
                    b.config(bg = "DarkOliveGreen1") #0,2,4 (all)
                
           
                b.grid(row=r, column=c)
                b.bind('<Button-3>', self.clickButton(r, c, 'R'))
                b.bind('<Button-1>', self.clickButton(r, c, 'L'))
         
                temp.append([b,0])
            self.mineFields.append(temp)

        print(self.mineFields)
        #[
        #[[b00,0][b01,2][b02,2]],
        #[[b10,1][b11,-1][b12,-1]],
        #[[b20,0][b21,2][b22,2]]
        #]
        
        #self.mineFields[r][c][0].config(relief = SUNKEN)
        #self.mineFields[5][1][0].config(relief = SUNKEN)

        # self.grid() positions and displays the current object instance (self) 
        # within its parent container using a two-dimensional grid
        self.grid()


        # Randomly convert 10% of the mineFields into mines
        self.createMines()
        # Update the number of mines around each cell.
        self.updateMineField()
        
        self.mainloop()
        

        
    def clickButton(self, r, c, clickType):

        def helper(event):
            print(clickType + " Click @ " + str(r) + ", " + str(c))

            # To display a window to indicate win or lose.
            # messagebox.showinfo("GG", "Win/Game over!")
        # self.mineFields[r][c][0]["text"] = "M"
        # self.mineFields[r][c][0].config(bg = "red")

            if clickType == 'L':
                if self.mineFields[r][c][0]["text"] == "🚩":
                    return #its a flag, dont open it
                
                if self.mineFields[r][c][0]["state"] == DISABLED and self.mineFields[r][c][0]["text"] != "🚩":
                    # self.mineFields[r][c][0].config(relief = RIDGE)
                    # self.mineFields[r][c][0].update()  
                    self.check_mines(r,c)

                    # self.mineFields[r][c][0].config(relief = SUNKEN)
                    # its disabled (either flag or number) and its not a flag (its a number)
                    # open surrounding
                    return

                if self.mineFields[r][c][1] != -1: #safe
                    self.openSafePos(r,c)
                else:
                    self.mineFields[r][c][0]["text"] = "M"
                    self.mineFields[r][c][0].config(bg = "red")
                    messagebox.showinfo("GG","Game Over")
                    # self.winfo_toplevel().destroy()    

            elif clickType == "R":
                print(self.mineFields[r][c][0]["state"])
                # check if its a flag alr, check that its not a number??
                if self.mineFields[r][c][0]["text"] == "":
                    self.mineFields[r][c][0]["text"] = "🚩"
                    self.mineFields[r][c][0].config(state = DISABLED)

                elif self.mineFields[r][c][0]["text"] == "🚩": 
                    print(self.mineFields[r][c][0]["text"],'text')
                    self.mineFields[r][c][0]["text"] = ""
                    self.mineFields[r][c][0].config(state = NORMAL)
                # self.mineFields[r][c][0].config(state = DISABLED)

            self.check_win()

            
            print(self.count,'count',(self.rows*self.cols),int(0.1 * (self.rows*self.cols)), (self.rows*self.cols) - int(0.1 * (self.rows*self.cols)))
            # if self.count == 0.9* (self.rows * self.cols):
            #     messagebox.showinfo("WIN!")
                # self.mineFields[r][c][0].config(relief = SUNKEN)
                # self.mineFields[r][c][0].config(state = DISABLED)
                # print(self.mineFields[r][c][1])
                # if self.mineFields[r][c][1] != -1: #safe
                #     self.openSafePos(r,c)
            

        return helper

    def openSafePos(self, r, c):
        # recursively open up all cells (by calling openSafePos) that do not have mines
        # that are connected to cell at (r,c)
        # print()
        print('opening',r,c)
        # if self.mine

        if self.mineFields[r][c][0]["state"] == DISABLED: #opened
            return 
        button = self.mineFields[r][c][0]
        val = self.mineFields[r][c][1]
        
        button.config(relief = SUNKEN,state = DISABLED)
        # button.config(bg = "khaki")
        if (r+c)%2 == 0:
            button.config(bg = "bisque")
        else:
            button.config(bg = "blanched almond")

        
        self.count += 1
        # if val isnt 0 then its a number and you should display
        if val != 0:
            button["text"] = str(val)
            return

     
        rows = self.rows
        cols = self.cols

        for dr, dc in self.neighbours:
            nr = r + dr
            nc = c + dc
            if (nr < 0) or (nr >= rows) or (nc < 0) or (nc >= cols):
                continue #out of valid range
            if self.mineFields[nr][nc][1] != -1:
                # self.mineFields[r][c][0]["text"] = self.mineFields[r][c][1]
                # self.count += 1
                self.openSafePos(nr,nc)
         
    def check_mines(self,r,c):
        num_mines = int(self.mineFields[r][c][1])


        rows = self.rows
        cols = self.cols
        mc = 0
        safes = []
        for dr, dc in self.neighbours:
            nr = r + dr
            nc = c + dc
            if (nr < 0) or (nr >= rows) or (nc < 0) or (nc >= cols):
                continue #out of valid range
            if self.mineFields[nr][nc][0]["text"] == "🚩":
                mc += 1
            else:
                safes.append([nr,nc])
        if mc == num_mines:
            for rr, cc in safes:
                if self.mineFields[rr][cc][1] == -1:
                    self.mineFields[rr][cc][0]["text"] = "M"
                    self.mineFields[rr][cc][0].config(bg="red")
                    messagebox.showinfo("GG", "Game Over")
                    # self.winfo_toplevel().destroy()
                    return
                else:
                    self.openSafePos(rr, cc)
        self.check_win()
            # self.winfo_toplevel().destroy()
            
    def createMines(self):
        # -1: mine
        # 0-8: number of mines around
        # Randomly convert 10% of the mineFields into mines
        # by changing the second element of the cell in the mineFields 2D list to -1
        # for example, self.mineFields[2][2][1] = -1
        # this means that the cell at (2,2) is a mine
        # print(self.mineFields)
        # print(sample(self.mineFields,3))
        # coords = []
        # for i in range(self.rows):
        #     for j in range(self.cols):
        #         coords.append((i,j))
        # print(coords)
        coords = [(i,j) for i in range(self.rows) for j in range(self.cols)]
        # print(coords)
        num_mines = int(0.1 * (self.rows*self.cols))
        # print(num_mines)
        mines = sample(coords,num_mines)
        print(mines,'sample mines')
        self.mines = mines
        for coord in mines:
            row = coord[0]
            col = coord[1]
            self.mineFields[row][col][1] = -1
            #update surrounding mines
            # neighbours = [
            #     [-1,-1],[-1,0],[-1,1],
            #     [0,-1],        [0,1],
            #     [1,-1],[-1,0],[1,1]
            # ]

            # top left: (r-1,c-1), top left: (r-1,c+1)
            # bottom left: (r+1,c-1), bottom right: (r+1,c+1)


    def updateMineField(self):
        # update the mineFields after mines are created
        # for example, self.mineFields[3][3][1] = 7
        # this means that the number of mines around the cell at (3,3) is 7

        # pass 
        rows = self.rows
        cols = self.cols
        # print(self.mines,'mines')
        for mine in self.mines:
            miner = mine[0]
            minec = mine[1]
            # print(miner,minec,'mine coord')
            for dr, dc in self.neighbours:
                nr  = dr + miner
                nc = dc + minec
                # print(nr,nc,(dr,dc))
                if (nr < 0) or (nr >= rows) or (nc < 0) or (nc >= cols):
                    continue #out of valid range
             
                # print(grid[nr][nc][1])
                if self.mineFields[nr][nc][1] == -1:
                    continue #its a mine
                else:
                    self.mineFields[nr][nc][1] += 1
                
                # self.mineFields[nr][nc][1] +

    def restart(self):
        self.destroy()
        mineSweeper(self.rows, self.cols)
    def check_win(self):
        if self.count == self.rows * self.cols - len(self.mines):
            messagebox.showinfo("GG", "WIN!")



mineSweeper(10,10)
# 0.1*25

[[[<tkinter.Button object .!minesweeper.!button3>, 0], [<tkinter.Button object .!minesweeper.!button4>, 0], [<tkinter.Button object .!minesweeper.!button5>, 0], [<tkinter.Button object .!minesweeper.!button6>, 0], [<tkinter.Button object .!minesweeper.!button7>, 0], [<tkinter.Button object .!minesweeper.!button8>, 0], [<tkinter.Button object .!minesweeper.!button9>, 0], [<tkinter.Button object .!minesweeper.!button10>, 0], [<tkinter.Button object .!minesweeper.!button11>, 0], [<tkinter.Button object .!minesweeper.!button12>, 0]], [[<tkinter.Button object .!minesweeper.!button13>, 0], [<tkinter.Button object .!minesweeper.!button14>, 0], [<tkinter.Button object .!minesweeper.!button15>, 0], [<tkinter.Button object .!minesweeper.!button16>, 0], [<tkinter.Button object .!minesweeper.!button17>, 0], [<tkinter.Button object .!minesweeper.!button18>, 0], [<tkinter.Button object .!minesweeper.!button19>, 0], [<tkinter.Button object .!minesweeper.!button20>, 0], [<tkinter.Button object .!mines

<__main__.mineSweeper object .!minesweeper>

In [ ]:
from random import sample

a = [[1,2],[3,4],[4,5]]
print(sample(a,2))

# [
# [[<tkinter.Button object .!minesweeper2.!button11>, 0], [<tkinter.Button object .!minesweeper2.!button12>, 0], 
#   [<tkinter.Button object .!minesweeper2.!button13>, 0], [<tkinter.Button object .!minesweeper2.!button14>, 0], 
#   [<tkinter.Button object .!minesweeper2.!button15>, 0]], 
# [[<tkinter.Button object .!minesweeper2.!button21>, 0], [<tkinter.Button object .!minesweeper2.!button22>, 0],
#     [<tkinter.Button object .!minesweeper2.!button23>, 0], [<tkinter.Button object .!minesweeper2.!button24>, 0], 
#     [<tkinter.Button object .!minesweeper2.!button25>, 0]], 
# [[<tkinter.Button object .!minesweeper2.!button16>, 0], [<tkinter.Button object .!minesweeper2.!button17>, 0], 
#     [<tkinter.Button object .!minesweeper2.!button18>, 0], [<tkinter.Button object .!minesweeper2.!button19>, 0],
#     [<tkinter.Button object .!minesweeper2.!button20>, 0]]
# ]
# # <__main__.mineSweeper object .!minesweeper2>

[[4, 5], [3, 4]]
